In [28]:
# Paste in Jupyter, paste output back
from pymongo import MongoClient
import os

uri = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
client = MongoClient(uri)
db = client[os.environ.get("MONGO_DATABASE", "quants_lab")]

pipeline = [
    {"$match": {"trading_pair": "XMR-USDT", "connector": {"$in": ["mexc", "nonkyc"]}}},
    {"$group": {
        "_id": {"connector": "$connector", "interval": "$interval"},
        "count": {"$sum": 1},
        "first_ts": {"$min": "$timestamp"},
        "last_ts":  {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.interval": 1}},
]
for doc in db["candles"].aggregate(pipeline):
    c = doc["_id"]["connector"]
    i = doc["_id"]["interval"]
    n = doc["count"]
    span_days = (doc["last_ts"] - doc["first_ts"]) / 86400
    print(f"{c:>8}  {i:>4}  {n:>7} bars   {span_days:6.1f} days")

    mexc   15m    34599 bars    360.4 days
    mexc    1d     2318 bars   2317.0 days
    mexc    1h     5410 bars    225.4 days
    mexc    1m   108748 bars     76.2 days
    mexc    4h     1351 bars    225.0 days
    mexc    5m   103802 bars    361.0 days
    mexc    8h      540 bars    179.7 days
  nonkyc   12h      360 bars    179.5 days
  nonkyc   15m    96889 bars   1053.5 days
  nonkyc    1d     1079 bars   1079.0 days
  nonkyc    1h    25541 bars   1079.5 days
  nonkyc    4h     6446 bars   1079.3 days
  nonkyc    5m   201414 bars    731.1 days
  nonkyc    8h      540 bars    179.7 days


In [1]:
# Diagnostic — inspect what NonKYC is returning for a pair that wrote 0
# Run after interrupting Cell 11 (click ⏹).

import time as _t

# Test a pair that previously "wrote 0 across 3 range(s)"
test_pair = "AAVE-USDT"
test_interval = "8h"

# Find the first gap for this pair
step = INTERVAL_SECONDS[test_interval]
win_start = utc_now_ts() - 365*86400  # last year
gaps = find_gaps_in_window(coll, "nonkyc", test_pair, test_interval, step,
                           win_start, utc_now_ts())

print(f"Gaps found for {test_pair} {test_interval}: {len(gaps)}")
for gs, ge in gaps[:3]:
    print(f"  {fmt_ts(gs)} → {fmt_ts(ge)}  ({(ge-gs)/86400:.1f} days)")

# Fetch one page directly and see what comes back
if gaps:
    gs, ge = gaps[0]
    print(f"\nFetching directly from NonKYC for first gap:")
    page_to = ge + step - 1
    _t0 = _t.perf_counter()
    bars = fetch_nonkyc_candles(test_pair, test_interval,
                                 to_ts=page_to, count=EXCHANGES["nonkyc"]["max_per_request"])
    _dt = _t.perf_counter() - _t0
    print(f"  Got {len(bars)} bars in {_dt*1000:.0f}ms")
    if bars:
        print(f"  First bar:  ts={bars[0]['timestamp']} ({fmt_ts(bars[0]['timestamp'])})")
        print(f"              o={bars[0]['open']} h={bars[0]['high']} l={bars[0]['low']} c={bars[0]['close']} v={bars[0]['volume']}")
        print(f"  Last bar:   ts={bars[-1]['timestamp']} ({fmt_ts(bars[-1]['timestamp'])})")
        print(f"              o={bars[-1]['open']} h={bars[-1]['high']} l={bars[-1]['low']} c={bars[-1]['close']} v={bars[-1]['volume']}")
        # How many bars fall in the requested gap window?
        in_window = sum(1 for b in bars if gs <= int(b["timestamp"]) <= ge)
        print(f"  Bars within gap window [{fmt_ts(gs)} → {fmt_ts(ge)}]: {in_window}/{len(bars)}")
        if in_window == 0:
            print(f"  ⚠  NO bars fell in the requested window — "
                  f"this is why 'wrote 0' happens!")
            print(f"     Returned range: {fmt_ts(bars[-1]['timestamp'])} → {fmt_ts(bars[0]['timestamp'])}")
            print(f"     Requested:      {fmt_ts(gs)} → {fmt_ts(ge)}")
    else:
        print(f"  NonKYC returned EMPTY — pair either didn't trade or has no data for this range")

# Also check how many candles already exist for this pair
count = coll.count_documents({"connector": "nonkyc", "trading_pair": test_pair,
                              "interval": test_interval})
print(f"\nMongo doc count for nonkyc {test_pair} {test_interval}: {count:,}")

NameError: name 'INTERVAL_SECONDS' is not defined

In [3]:
import sys, threading, traceback
frames = sys._current_frames()
for t in threading.enumerate():
    if "ThreadPoolExecutor" in t.name:
        frame = frames.get(t.ident)
        if frame:
            stack = traceback.extract_stack(frame)
            print(f"{t.name}:")
            for entry in stack[-4:]:
                print(f"  {entry.filename.split('/')[-1]}:{entry.lineno} in {entry.name}")
                if entry.line:
                    print(f"    {entry.line}")
            print()